# 🎮 Task 2 — RAWG Video Games Database API

This notebook consumes the [RAWG API](https://rawg.io/apidocs) to extract, analyze, and compare video game data using Python.

**Author:** resaca_2014  
**Date:** 2026-04-10

## ⚙️ Setup — Imports & API Configuration

We import the required libraries and define the API key and base URL.  
The API key is stored in a local variable and **must not** be uploaded to GitHub.

In [1]:
import requests
import pandas as pd
import os

# ⚠️ Store your API key locally — do NOT commit this value
API_KEY = "3290a5edd51b45218a2a18bfc752f4ac"
BASE_URL = "https://api.rawg.io/api"

def get(endpoint, params=None):
    """Helper function to make GET requests to the RAWG API."""
    if params is None:
        params = {}
    params["key"] = API_KEY
    response = requests.get(f"{BASE_URL}{endpoint}", params=params)
    response.raise_for_status()
    return response.json()

print("Setup complete. Ready to query RAWG API.")

Setup complete. Ready to query RAWG API.


---
## 🟢 Part A — General Exploration

### A1 — Total number of games registered in RAWG

We query the `/games` endpoint and read the `count` field from the response, which tells us the total number of games indexed by RAWG.

In [2]:
# A1 — Total games registered in RAWG
# We only need 1 result; the 'count' field always reflects the full total
data = get("/games", {"page_size": 1})
total_games = data["count"]
print(f"🎮 Total games registered in RAWG: {total_games:,}")

🎮 Total games registered in RAWG: 898,481


---
## 🔵 Part B — Category Analysis

### B1 — Top 5 highest rated games of all time (by Metacritic)

We sort by `metacritic` score in descending order to find the best critically acclaimed games ever registered in RAWG.

In [3]:
# B1 — Top 5 games by Metacritic score
data = get("/games", {
    "ordering": "-metacritic",  # orden descendente por puntaje Metacritic
    "page_size": 5
})

top5_metacritic = pd.DataFrame([
    {
        "name":       g["name"],
        "rating":     g["rating"],
        "metacritic": g["metacritic"]
    }
    for g in data["results"]
])

print("🏆 Top 5 Highest Rated Games of All Time (Metacritic):")
display(top5_metacritic)

🏆 Top 5 Highest Rated Games of All Time (Metacritic):


,name,rating,metacritic
0,The Legend of Zelda: Ocarina of Time,4.38,99
1,Soulcalibur (1998),0.00,98
2,Soulcalibur,4.38,98
3,Baldur's Gate III,4.44,97
4,Metroid Prime,4.35,97


### B2 — Top 10 best games available on Steam (store_id=1)

We filter games available on Steam using its store ID and sort by user rating to find the best-reviewed Steam games.

In [5]:
# B2 — Top 10 games on Steam
data = get("/games", {
    "stores": 1,            # filtra solo juegos disponibles en Steam (store_id=1)
    "ordering": "-rating",  # ordena de mayor a menor por rating de usuarios
    "page_size": 10
})

top10_steam = pd.DataFrame([
    {
        "name":       g["name"],
        "rating":     g["rating"],
        "metacritic": g["metacritic"]
    }
    for g in data["results"]
])

print("🖥️ Top 10 Best Games on Steam:")
display(top10_steam)

🖥️ Top 10 Best Games on Steam:


,name,rating,metacritic
0,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN
1,No Case Should Remain Unsolved,4.83,NaN
2,The Witcher 3: Wild Hunt – Blood and Wine,4.81,92.0
3,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0
4,The Witcher 3: Wild Hunt – Hearts of Stone,4.76,90.0
5,Persona 5 Royal,4.75,94.0
6,Guilty Parade,4.71,NaN
7,Cyberpunk 2077: Phantom Liberty,4.71,NaN
8,The Binding of Isaac: Repentance,4.69,NaN
9,Red Matter 2,4.67,NaN


---
## 🟡 Part C — Comparisons

### C1 — PC (platform_id=4) vs PS5 (platform_id=187)

We query the top 5 games for each platform sorted by user rating, then compare their averages to determine which platform has the highest rated games.

In [9]:
# C1 — PC vs PS5 top 5 games comparison

def top5_by_platform(platform_id, platform_name):
    """Fetches top 5 games for a given platform sorted by user rating."""
    data = get("/games", {
        "platforms": platform_id,  # filtra por plataforma
        "ordering": "-rating",     # ordena de mayor a menor por rating
        "page_size": 5
    })
    df = pd.DataFrame([
        {
            "name":       g["name"],
            "rating":     g["rating"],
            "metacritic": g["metacritic"]
        }
        for g in data["results"]
    ])
    df["platform"] = platform_name  # agrega columna con el nombre de la plataforma
    return df

# Consultamos ambas plataformas
pc_top5  = top5_by_platform(4,   "PC")
ps5_top5 = top5_by_platform(187, "PS5")

# Mostramos cada plataforma por separado
print("🖥️ Top 5 Best Games on PC:")
display(pc_top5)

print("🎮 Top 5 Best Games on PS5:")
display(ps5_top5)

# Calculamos el promedio de rating por plataforma
comparison = pd.concat([pc_top5, ps5_top5], ignore_index=True)
avg_ratings = comparison.groupby("platform")["rating"].mean()
best_platform = avg_ratings.idxmax()

print(f"\n📊 Average rating by platform:")
display(avg_ratings.reset_index().rename(columns={"rating": "avg_rating"}))
print(f"\n🏆 Platform with highest rated games: {best_platform}")

🖥️ Top 5 Best Games on PC:


,name,rating,metacritic,platform
0,The Elder Scrolls VI,4.86,NaN,PC
1,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN,PC
2,No Case Should Remain Unsolved,4.83,NaN,PC
3,The Witcher 3: Wild Hunt – Blood and Wine,4.81,92.0,PC
4,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0,PC


🎮 Top 5 Best Games on PS5:


,name,rating,metacritic,platform
0,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0,PS5
1,Persona 5 Royal,4.75,94.0,PS5
2,Cyberpunk 2077: Phantom Liberty,4.71,NaN,PS5
3,The Binding of Isaac: Repentance,4.69,NaN,PS5
4,The Last of Us Part I,4.67,NaN,PS5



📊 Average rating by platform:


,platform,avg_rating
0,PC,4.826
1,PS5,4.724



🏆 Platform with highest rated games: PC


### C2 — Comparison table of 3 famous games

We search for 3 iconic titles and build a side-by-side comparison table showing name, rating, metacritic score, genres, and available platforms.

In [ ]:
# C2 — Comparison table of 3 famous games
famous_games = ["Tom Clancy's The Division 2", "Metal Gear Solid V: The Phantom Pain", "Tom Clancy's Splinter Cell Blacklist"]

rows = []
for title in famous_games:
    # Buscamos cada juego por nombre y tomamos el primer resultado
    data = get("/games", {"search": title, "page_size": 1})
    g = data["results"][0]
    rows.append({
        "name":       g["name"],
        "rating":     g["rating"],
        "metacritic": g["metacritic"],
        # Extraemos los géneros y plataformas como texto separado por comas
        "genres":     ", ".join([genre["name"] for genre in g["genres"]]),
        "platforms":  ", ".join([p["platform"]["name"] for p in g["platforms"]])
    })

comparison_table = pd.DataFrame(rows)

print("🎮 Famous Games Comparison Table:")
display(comparison_table)